# Section 5: Adaptive inflation

*(Replaces DART_LAB slide deck Section 5.)*

Section 3 tuned a fixed inflation value by hand. Real observing systems
change in time and space, so a single hand-tuned number is rarely right.
**Adaptive inflation** treats the inflation $\lambda$ itself as a random
variable and updates it with Bayes' rule using each observation.

In [ ]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import pydartlab as dl
import pydartlab.apps as apps

## The idea

If the prior (inflated by $\lambda$) and the observation are consistent,
the expected squared separation is

$$ E[d^2] = \theta^2 = \lambda \sigma_p^2 + \sigma_o^2 . $$

So the actual innovation $d = |y_o - \bar{x}_p|$ carries information about
$\lambda$: the likelihood is
$p(y_o \mid \lambda) = (2\pi\theta^2)^{-1/2} \exp(-d^2 / 2\theta^2)$.
Combine it with a prior $p(\lambda) = N(\bar\lambda, \sigma_\lambda^2)$
(or an inverse-gamma, which respects $\lambda > 0$) and you get a posterior
for the inflation. DART implements two flavors:

* **Gaussian** — Anderson (2009),
* **Inverse-Gamma** — El Gharamti (2018), now the DART default.

The cell below shows one Bayesian update of $\lambda$ for a small and a
large innovation.

In [ ]:
from pydartlab.algorithms.inflation import compute_new_density

lam = np.linspace(0.5, 4, 300)
sigma_p2, sigma_o2 = 1.0, 1.0
fig, axes = plt.subplots(1, 2, figsize=(9, 3), sharey=True)
for ax, d in zip(axes, (0.5, 4.0)):
    dens = [compute_new_density(d**2, sigma_p2, sigma_o2, 1.0, 0.6, 1.0, la)
            for la in lam]
    ax.plot(lam, np.array(dens) / np.max(dens))
    new_mean, new_sd = dl.update_inflate(0.0, sigma_p2, d, sigma_o2, 1.0,
                                         1.0, 0.6, 1.0, 100.0, 1.0, 0.1, 20,
                                         "Gaussian")
    ax.axvline(new_mean, color="C1")
    ax.set_title(f"innovation d = {d}: new mean = {new_mean:.2f}")
    ax.set_xlabel(r"inflation $\lambda$")
axes[0].set_ylabel("posterior density (normalized)");

A small innovation leaves the inflation near 1 (clamped at the lower
bound); a large innovation — the ensemble was overconfident — pushes it up.

The controls every adaptive scheme exposes:

* **lower/upper bounds** on the inflation mean (lower bound < 1 permits
  *deflation*),
* the **inflation SD** (how fast it can adapt) and its lower bound,
* **damping**: each cycle, `inflation = 1 + damping * (inflation - 1)`
  relaxes the field toward 1, useful when the observing network changes.

## Exercise: `oned_model_inf`

1. Select **Adaptive Inflation** and add **Bias = 1**: watch the inflation
   panel — $\lambda$ rises until error and spread are consistent, with no
   hand tuning.
2. Lower the **Inf SD** (e.g. 0.3): adaptation is slower but steadier.
3. Set the lower bound below 1 (e.g. 0.5) and remove the bias: can you see
   deflation?
4. Compare with damping 0.9 vs 1.0.

In [ ]:
omi = apps.oned_model_inf(seed=3)
omi

In [ ]:
# Scripted: adaptive inflation discovers the bias, fixed inflation must be told
from pydartlab.experiments import OneDExperiment

fig, ax = plt.subplots(figsize=(7, 3))
exp = OneDExperiment(ens_size=10, model_bias=1.0, adaptive_inflation=True,
                     inflation_max=10.0, seed=8)
for _ in range(300):
    exp.step()
ax.plot(exp.history["inflation"])
ax.set_xlabel("cycle"); ax.set_ylabel("inflation mean")
ax.set_title("Adaptive inflation responding to model bias");

## Spatially varying inflation: `run_lorenz_96_inf`

In a spatial model each state variable gets its own $\lambda_i$, updated
using the localized correlation between it and each observation. Regions
with model error or poor observation coverage develop higher inflation.

## Exercises

1. **Adaptive Inflation** + localization 0.2; run a few hundred steps and
   watch the inflation field panel.
2. Create **model error**: set Forcing = 6 or 10 (truth stays at 8). How
   does the inflation field respond? Do good localization values change?
3. Change the **observing network** (e.g. `1:20` observes only the first
   half of the domain): inflation grows in the unobserved region.
4. Switch the flavor between **Gaussian** and **I-Gamma** — the
   inverse-gamma flavor respects the bound at zero and is DART's default.

In [ ]:
l96i = apps.run_lorenz_96_inf(seed=4)
l96i

In [ ]:
# Scripted: inflation field after 150 steps observing only variables 1:20
from pydartlab.experiments import Lorenz96Experiment

exp = Lorenz96Experiment(filter_type="EAKF", localization=0.2,
                         adaptive_inflation=True, inflation_damping=1.0,
                         obs_error_sd=1.0, obs_network="1:20", seed=12)
for _ in range(150):
    exp.step()
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(exp.inflate, "-x")
ax.axvspan(1, 20, alpha=0.12, color="green", label="observed region")
ax.axhline(1.0, color="0.7")
ax.set_xlabel("state variable"); ax.set_ylabel("inflation")
ax.legend(); ax.set_title("Inflation adapts to the observing network");

## What you should have seen

* Each observation carries information about whether the ensemble spread
  is right; Bayes' rule turns that into an inflation update.
* Adaptive inflation finds, without tuning, roughly the value you found by
  hand in Section 3 — and varies it in space when the observing network or
  model error is inhomogeneous.

**Next: Section 6 — everything in this tutorial, in the real DART
system.**